# Week 2: Model Development - AI Health Predictor

This notebook covers the model development phase for the AI Health Predictor project.

In [ ]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, auc
)
import warnings
warnings.filterwarnings('ignore')

print("Essential libraries imported successfully!")

In [ ]:
# Check for existing data
import os

print("Looking for existing datasets...")
data_files = []
for root, dirs, files in os.walk('.'):
    for file in files:
        if file.endswith('.csv'):
            data_files.append(os.path.join(root, file))

if data_files:
    print(f"Found {len(data_files)} CSV files:")
    for file in data_files[:5]:
        print(f"  - {file}")
    
    # Load the first CSV file
    df = pd.read_csv(data_files[0])
    print(f"\nLoaded dataset: {data_files[0]}")
else:
    print("No CSV files found. Creating sample diabetes dataset...")
    np.random.seed(42)
    n_samples = 768
    df = pd.DataFrame({
        'Pregnancies': np.random.randint(0, 10, n_samples),
        'Glucose': np.random.randint(50, 200, n_samples),
        'BloodPressure': np.random.randint(60, 120, n_samples),
        'SkinThickness': np.random.randint(20, 50, n_samples),
        'Insulin': np.random.randint(0, 300, n_samples),
        'BMI': np.round(np.random.uniform(18, 40, n_samples), 1),
        'DiabetesPedigreeFunction': np.round(np.random.uniform(0.1, 1.5, n_samples), 3),
        'Age': np.random.randint(20, 70, n_samples),
        'Outcome': np.random.randint(0, 2, n_samples)
    })

print(f"\nDataset shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())

In [ ]:
# Data Exploration
print("=== DATA EXPLORATION ===\n")

# Basic info
print("1. Dataset Information:")
print(df.info())

# Missing values
print("\n2. Missing Values:")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("No missing values found.")

# Statistical summary
print("\n3. Statistical Summary:")
print(df.describe())

# Identify target variable
target_candidates = ['Outcome', 'target', 'diagnosis', 'result', 'class']
target_col = None
for col in target_candidates:
    if col in df.columns:
        target_col = col
        break
if target_col is None:
    target_col = df.columns[-1]
    
print(f"\n4. Target Variable: '{target_col}'")
print(f"   Unique values: {df[target_col].unique()}")
print(f"   Value counts:\n{df[target_col].value_counts()}")

# Visualize target distribution
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
df[target_col].value_counts().plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Target Distribution')
plt.xlabel('Class')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
df[target_col].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['lightblue', 'lightcoral'])
plt.title('Target Percentage')
plt.ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Data Preparation
print("=== DATA PREPARATION ===\n")

# Handle missing values
if df.isnull().sum().sum() > 0:
    print(f"Filling {df.isnull().sum().sum()} missing values with median...")
    df = df.fillna(df.median())

# Separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

# Encode target if needed
if y.dtype == 'object':
    print("Encoding target variable...")
    le = LabelEncoder()
    y = le.fit_transform(y)
    print(f"Classes: {le.classes_}")

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature names: {X.columns.tolist()}")

In [ ]:
# Feature Engineering
print("=== FEATURE ENGINEERING ===\n")

# Create interaction features
numeric_cols = X.select_dtypes(include=[np.number]).columns
if len(numeric_cols) >= 2:
    col1, col2 = numeric_cols[0], numeric_cols[1]
    X['interaction_feature'] = X[col1] * X[col2]
    print(f"Created interaction feature: {col1} * {col2}")

# Create polynomial features
if len(numeric_cols) > 0:
    first_col = numeric_cols[0]
    X[f'{first_col}_squared'] = X[first_col] ** 2
    print(f"Created squared feature for {first_col}")

print(f"\nNew features shape: {X.shape}")
print(f"New feature names: {X.columns.tolist()}")

In [ ]:
# Train-Test Split
print("=== TRAIN-TEST SPLIT ===\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures scaled using StandardScaler")

## Phase 1: Baseline Models

In [ ]:
# Train Baseline Models
print("=== BASELINE MODELS ===\n")

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # ROC-AUC if probabilities are available
    if hasattr(model, 'predict_proba'):
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
        roc_auc = roc_auc_score(y_test, y_pred_proba)
    else:
        roc_auc = None
    
    # Store results
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc
    })
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    if roc_auc:
        print(f"  ROC-AUC: {roc_auc:.4f}")
    print()

# Convert to DataFrame
results_df = pd.DataFrame(results)
print("\n=== MODEL PERFORMANCE ===")
print(results_df.to_string())

In [ ]:
# Visualize Model Performance
print("=== VISUALIZATION ===\n")

plt.figure(figsize=(14, 6))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['skyblue', 'lightgreen', 'salmon', 'gold']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    plt.subplot(1, 4, i+1)
    plt.bar(results_df['Model'], results_df[metric], color=color)
    plt.title(metric)
    plt.xticks(rotation=45)
    plt.ylim(0, 1)
    
    # Add value labels
    for j, value in enumerate(results_df[metric]):
        plt.text(j, value + 0.02, f'{value:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# Save the plot
plt.savefig('week2/reports/model_comparison.png', dpi=150, bbox_inches='tight')
print("Model comparison plot saved to week2/reports/model_comparison.png")

In [ ]:
# Identify Best Model
print("=== BEST MODEL SELECTION ===\n")

best_idx = results_df['Accuracy'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
best_model = models[best_model_name]
best_accuracy = results_df.loc[best_idx, 'Accuracy']

print(f"Best Model: {best_model_name}")
print(f"Accuracy: {best_accuracy:.4f}")
print(f"Precision: {results_df.loc[best_idx, 'Precision']:.4f}")
print(f"Recall: {results_df.loc[best_idx, 'Recall']:.4f}")
print(f"F1-Score: {results_df.loc[best_idx, 'F1-Score']:.4f}")

if results_df.loc[best_idx, 'ROC-AUC']:
    print(f"ROC-AUC: {results_df.loc[best_idx, 'ROC-AUC']:.4f}")

## Phase 2: Model Evaluation

In [ ]:
# Detailed Evaluation of Best Model
print(f"=== DETAILED EVALUATION: {best_model_name} ===\n")

# Get predictions from best model
y_pred = best_model.predict(X_test_scaled)
y_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1] if hasattr(best_model, 'predict_proba') else None

# Confusion Matrix
print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ROC Curve if probabilities available
if y_pred_proba is not None:
    print("\nGenerating ROC Curve...")
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {best_model_name}')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Save ROC curve
    plt.savefig('week2/reports/roc_curve.png', dpi=150, bbox_inches='tight')
    print("ROC curve saved to week2/reports/roc_curve.png")

## Phase 3: Save Models and Reports

In [ ]:
# Save Models and Create Reports
print("=== SAVING MODELS AND REPORTS ===\n")

import joblib
import json
from datetime import datetime

# Create directories
import os
os.makedirs('week2/models', exist_ok=True)
os.makedirs('week2/reports', exist_ok=True)

# 1. Save scaler
joblib.dump(scaler, 'week2/models/scaler.pkl')
print("✓ Scaler saved: week2/models/scaler.pkl")

# 2. Save all models
for name, model in models.items():
    filename = f"week2/models/{name.lower().replace(' ', '_')}.pkl"
    joblib.dump(model, filename)
    print(f"✓ {name} saved: {filename}")

# 3. Save best model separately
joblib.dump(best_model, 'week2/models/best_model.pkl')
print(f"✓ Best model ({best_model_name}) saved: week2/models/best_model.pkl")

# 4. Save feature names
feature_names = X.columns.tolist()
with open('week2/models/feature_names.json', 'w') as f:
    json.dump(feature_names, f)
print("✓ Feature names saved: week2/models/feature_names.json")

# 5. Create performance report
report = {
    'timestamp': datetime.now().isoformat(),
    'dataset': {
        'original_shape': df.shape,
        'training_samples': len(X_train),
        'testing_samples': len(X_test),
        'target_variable': target_col,
        'features_count': len(feature_names)
    },
    'best_model': {
        'name': best_model_name,
        'accuracy': float(best_accuracy),
        'precision': float(results_df.loc[best_idx, 'Precision']),
        'recall': float(results_df.loc[best_idx, 'Recall']),
        'f1_score': float(results_df.loc[best_idx, 'F1-Score'])
    },
    'all_models': results_df.to_dict('records')
}

with open('week2/reports/performance_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print("✓ Performance report saved: week2/reports/performance_report.json")

# 6. Save predictions
predictions_df = pd.DataFrame({
    'actual': y_test,
    'predicted': y_pred,
    'probability': y_pred_proba if y_pred_proba is not None else [None] * len(y_test)
})
predictions_df.to_csv('week2/reports/test_predictions.csv', index=False)
print("✓ Test predictions saved: week2/reports/test_predictions.csv")

# 7. Save confusion matrix
cm_df = pd.DataFrame(cm, 
                     index=['Actual Negative', 'Actual Positive'],
                     columns=['Predicted Negative', 'Predicted Positive'])
cm_df.to_csv('week2/reports/confusion_matrix.csv')
print("✓ Confusion matrix saved: week2/reports/confusion_matrix.csv")

print("\n" + "="*60)
print("WEEK 2 COMPLETED SUCCESSFULLY!")
print("="*60)
print(f"\nSummary:")
print(f"- Best Model: {best_model_name}")
print(f"- Accuracy: {best_accuracy:.4f}")
print(f"- Models saved: {len(models)} models")
print(f"- Reports generated: 5 reports")
print(f"\nFiles saved in 'week2/' directory:")
print("  models/: Contains all trained models")
print("  reports/: Contains performance reports and visualizations")